# 语义分割和数据集
:label:`sec_semantic_segmentation`

在 :numref:`sec_bbox`— :numref:`sec_rcnn`中讨论的目标检测问题中，我们一直使用方形边界框来标注和预测图像中的目标。
本节将探讨*语义分割*（semantic segmentation）问题，它重点关注于如何将图像分割成属于不同语义类别的区域。
与目标检测不同，语义分割可以识别并理解图像中每一个像素的内容：其语义区域的标注和预测是像素级的。
 :numref:`fig_segmentation`展示了语义分割中图像有关狗、猫和背景的标签。
与目标检测相比，语义分割标注的像素级的边框显然更加精细。

![语义分割中图像有关狗、猫和背景的标签](../img/segmentation.svg)
:label:`fig_segmentation`

## 图像分割和实例分割

在计算机视觉领域，图像分割任务根据**处理粒度**和**目标侧重**的不同，主要分为三大核心方向：**图像分割（传统/无监督 segmentation）**、**语义分割（Semantic Segmentation）** 以及 **实例分割（Instance Segmentation）**。

---

### 1. 图像分割（Traditional / Unsupervised Image Segmentation）

图像分割是最基础、最传统的低阶视觉任务。找到的是封闭的、有意义的区域边界，把物体从背景中完整“抠”出来

* **核心定义**：基于图像的底层物理特征（如颜色、灰度、纹理、边缘梯度等），将图像划分为若干个“视觉上连贯且互不重叠”的区域。
* **机制特点**：
* **完全无监督**：不需要任何人工标注信息，不依赖深度学习数据驱动。
* **无语义概念（No Semantics）**：算法对图像中的物体毫无认知。例如，一只黑白相间的猫，算法会因为颜色断层，把“黑色毛发”和“白色毛发”强行切分成两个独立的区域。


* **代表算法**：分水岭算法（Watershed）、K-Means 聚类分割、Graph-Cut、Mean-Shift、Otsu 阈值分割。

---

### 2. 语义分割（Semantic Segmentation）

语义分割是引入了深度学习之后的**像素级分类任务**。也就是给每个像素点都做分类

* **核心定义**：将图像中的每一个像素点都打上对应的**类别标签（Class Label）**，实现“按类分块”。
* **机制特点**：
* **有监督学习**：依赖带像素级类别标注的数据集进行训练。
* **区分语义，不区分个体**：如果图像中有三个人，语义分割会将这三个人占据的所有像素统统标记为 `Person` 类（在可视化时渲染为同一种颜色），无法判断某个像素具体属于“张三”还是“李四”。
* **全图覆盖（Things + Stuff）**：通常会对整张图的每个像素（包括天空、路面、建筑物等没有固定轮廓的背景背景/Stuff）都进行预测。


* **代表算法**：FCN、DeepLab 系列、PSPNet、SegNet、Mask2Former。

---

### 3. 实例分割（Instance Segmentation）

实例分割是**目标检测（Object Detection）与语义分割的复合任务**（Simultaneous Detection and Segmentation）。

* **核心定义**：不仅要识别出像素所属的语义类别，还要精确区分出**属于同一个类别的不同物体个体（Instance ID）**。
* **机制特点**：
* **高阶深度检测 + 分割**：通常先通过框（BBox）定位出目标个体的区域，再在框内预测像素掩码（Mask）。
* **既分语义，又分个体**：同样是三个人，实例分割会分别输出 `Person #1`、`Person #2` 和 `Person #3`，并用三种不同的颜色掩码独立框选。
* **聚焦可数物体（Things）**：主要针对有明确几何轮廓、可数的独立目标（如人、车、动物），一般不处理天空、水流等连续背景。


* **代表算法**：Mask R-CNN、YOLACT、SOLO、CenterMask。

---

### 📊 三种分割任务全面对比表

| 比较维度 | 图像分割 (Traditional) | 语义分割 (Semantic) | 实例分割 (Instance) |
| --- | --- | --- | --- |
| **核心本质** | 底层物理特征聚类 | 像素级多分类 | 目标检测 + 像素级掩码预测 |
| **驱动方式** | **无监督**（手工设计规则） | **有监督**（深度学习） | **有监督**（深度学习） |
| **语义识别能力** | ❌ **无**（不知物体为何物） | **有**（识别像素所属类别） | **有**（识别像素所属类别） |
| **个体区分能力** | ❌ **无**（按颜色/纹理切碎） | ❌ **无**（同类像素连成一片） | **有**（精确区分同一类别的个体 A 和 B） |
| **标注依赖** | 无需任何数据集和标注 | 需要像素级类别 Mask 标注 | 需要 BBox 边界框 + 像素级 Instance Mask 标注 |
| **主要处理对象** | 全图局部连续区域 | 全图所有像素（含前景物体与连续背景） | **可数的独立个体**（如车辆、行人、动物等） |
| **输出形式** | 若干无语义的区域掩码 | 与原图同尺寸的**类别矩阵** $[H, W]$ | 目标检测框 $[N, 4]$ + 独立掩码矩阵 $[N, H, W]$ |
| **典型应用场景** | 医学图像预处理、工业缺陷检测 | 无人驾驶路况理解、遥感地块识别 | 机器人抓取、人像抠图/特效、智能安防跟踪 |
| **代表性算法** | K-Means, Graph-Cut, Watershed | FCN, DeepLab, PSPNet | Mask R-CNN, YOLACT, SOLO |


**实例分割（Instance Segmentation）** 的核心特点可以总结为以下 5 个维度：

---

### 1. 核心定位：目标检测与语义分割的“复合体”

* **公式化表达**：$\text{实例分割} = \text{目标检测（定位与区分个体）} + \text{语义分割（像素级精准抠图）}$。
* **分而治之**：先利用目标检测的矩形框（BBox）锁定物体的个体身份与大致位置，再在框内部进行像素级的二值分割（前景 vs 背景）。

---

### 2. 粒度精细：像素级的非规则描边（Mask）

* **摆脱粗暴矩形框**：相比目标检测只给出一个包围框，实例分割能沿着物体的真实边缘生成任意不规则形态的二值掩码（Mask）。
* **剔除背景噪声**：精准扣除框内的背景杂质与邻居像素，获得纯粹的物体像素集合，便于精确计算物体的面积、体积或轮廓。

---

### 3. 数据结构：基于对象档案的“实例级解耦”

* **代码独立存储**：每一个预测到的物体，在模型输出中都是一个包含 `[Instance ID, Class, Bounding Box, Mask]` 的独立数据对象/结构体。
* **区分同类个体**：即使同类物体高度重叠或交叠，因为各自拥有独立的 ID、检测框和 Mask 数组，因此可以实现高强度的个体区分与轨迹追踪。

---

### 4. 关注对象：专精于“可数实体”（Things）

* **聚焦 Things**：主要针对具有明确几何轮廓、可以独立计数的物体（如行人、车辆、动物、细胞、工业零件等）。
* **不处理连续背景**：通常不处理天空、水体、路面等没有固定边界且不可数的背景（Stuff）。

---

### 💡 终极一句话区别

* **目标检测**：区分了个体，但只给了**粗暴的方框**（框内混杂背景）。
* **语义分割**：精准到了像素，但**丢失了个体概念**（同类连成一片）。
* **实例分割**：既精准到了**像素级 Mask**，又完美保留了**个体身份（ID）**。

## Pascal VOC2012 语义分割数据集

[**最重要的语义分割数据集之一是[Pascal VOC2012](http://host.robots.ox.ac.uk/pascal/VOC/voc2012/)。**]
下面我们深入了解一下这个数据集。


In [ ]:
# 导入必要的库
%matplotlib inline
import os
import torch
import torchvision
from d2l import torch as d2l

数据集的tar文件大约为2GB，所以下载可能需要一段时间。
提取出的数据集位于`../data/VOCdevkit/VOC2012`。


In [ ]:
# 配置 VOC2012 数据集的下载信息
# @save 标记表示这个函数/变量在书中会被保存以便复用
#@save
d2l.DATA_HUB['voc2012'] = (d2l.DATA_URL + 'VOCtrainval_11-May-2012.tar',
                           '4e443f8a2eca6b1dac8a6c57641b67dd40621a49')

# 下载并提取数据集，返回数据集根目录路径
# VOCdevkit/VOC2012 是 tar 包解压后的子目录路径
voc_dir = d2l.download_extract('voc2012', 'VOCdevkit/VOC2012')

进入路径`../data/VOCdevkit/VOC2012`之后，我们可以看到数据集的不同组件。
`ImageSets/Segmentation`路径包含用于训练和测试样本的文本文件，而`JPEGImages`和`SegmentationClass`路径分别存储着每个示例的输入图像和标签。
此处的标签也采用图像格式，其尺寸和它所标注的输入图像的尺寸相同。
此外，标签中颜色相同的像素属于同一个语义类别。
下面将`read_voc_images`函数定义为[**将所有输入的图像和标签读入内存**]。


In [ ]:
#@save
def read_voc_images(voc_dir, is_train=True):
    """读取所有 VOC 图像并标注
    
    参数:
        voc_dir: VOC 数据集根目录路径
        is_train: True 读取训练集，False 读取验证集
        
    返回:
        features: 图像列表，每个元素是形状为 [C, H, W] 的张量，C=3(RGB)
        labels: 标注列表，每个元素是形状为 [C, H, W] 的张量，C=3(RGB 颜色编码的类别)
    """
    # 读取 train.txt 或 val.txt，文件中每行是一个图像的序号（不含扩展名）
    txt_fname = os.path.join(voc_dir, 'ImageSets', 'Segmentation',
                             'train.txt' if is_train else 'val.txt')
    # 设置图像读取模式为 RGB 三通道
    mode = torchvision.io.image.ImageReadMode.RGB
    with open(txt_fname, 'r') as f:
        images = f.read().split()  # 按空白字符分割，得到所有图像序号列表
    
    features, labels = [], []
    for i, fname in enumerate(images):
        # 读取 JPEG 图像，返回形状 [3, H, W]，dtype=uint8 (0-255)
        features.append(torchvision.io.read_image(os.path.join(
            voc_dir, 'JPEGImages', f'{fname}.jpg')))
        # 读取分割标注 PNG，也是 [3, H, W]，但每个像素的 RGB 值代表类别（颜色编码）
        labels.append(torchvision.io.read_image(os.path.join(
            voc_dir, 'SegmentationClass' ,f'{fname}.png'), mode))
    return features, labels

# 读取训练集的所有图像和标注
train_features, train_labels = read_voc_images(voc_dir, True)

下面我们[**绘制前5个输入图像及其标签**]。
在标签图像中，白色和黑色分别表示边框和背景，而其他颜色则对应不同的类别。


In [ ]:
# 绘制前 n=5 个样本的输入图像和对应的标注
n = 5
# 将 5 个输入图像和 5 个标注拼接成 10 个图像
# train_features[0:n]: 形状为 [5, 3, H, W] 的图像列表
# train_labels[0:n]: 形状为 [5, 3, H, W] 的标注列表
imgs = train_features[0:n] + train_labels[0:n]
# 转换通道顺序：从 [C, H, W] 变为 [H, W, C]，因为 matplotlib 显示图像需要 HWC 格式
# permute(1,2,0) 表示原第 1 维(H)→新第 0 位，原第 2 维(W)→新第 1 位，原第 0 维(C)→新第 2 位
imgs = [img.permute(1,2,0) for img in imgs]
# 显示图像：2 行 n 列，共 10 张图（前 5 张是原图，后 5 张是标注）
d2l.show_images(imgs, 2, n);

接下来，我们[**列举RGB颜色值和类名**]。


In [ ]:
# VOC2012 数据集的 21 个类别对应的 RGB 颜色值
# 每个类别在标注图像中用特定的 RGB 颜色表示
# 例如：[0, 0, 0] 黑色表示背景，[128, 0, 0] 表示飞机
#@save
VOC_COLORMAP = [[0, 0, 0], [128, 0, 0], [0, 128, 0], [128, 128, 0],
                [0, 0, 128], [128, 0, 128], [0, 128, 128], [128, 128, 128],
                [64, 0, 0], [192, 0, 0], [64, 128, 0], [192, 128, 0],
                [64, 0, 128], [192, 0, 128], [64, 128, 128], [192, 128, 128],
                [0, 64, 0], [128, 64, 0], [0, 192, 0], [128, 192, 0],
                [0, 64, 128]]

# 21 个语义类别的名称列表
# 索引 0 是背景 (background)，索引 1 是飞机 (aeroplane)，以此类推
#@save
VOC_CLASSES = ['background', 'aeroplane', 'bicycle', 'bird', 'boat',
               'bottle', 'bus', 'car', 'cat', 'chair', 'cow',
               'diningtable', 'dog', 'horse', 'motorbike', 'person',
               'potted plant', 'sheep', 'sofa', 'train', 'tv/monitor']

通过上面定义的两个常量，我们可以方便地[**查找标签中每个像素的类索引**]。
我们定义了`voc_colormap2label`函数来构建从上述RGB颜色值到类别索引的映射，而`voc_label_indices`函数将RGB值映射到在Pascal VOC2012数据集中的类别索引。


In [ ]:
#@save
def voc_colormap2label():
    """构建从 RGB 到 VOC 类别索引的映射
    
    返回:
        colormap2label: 形状为 [256^3] 的一维张量，即长度为 16777216 的查找表
        对于每个 RGB 颜色 (R, G, B)，可以通过索引 (R*256 + G)*256 + B 查找到对应的类别索引
    """
    # 创建一个长度为 256^3 = 16777216 的查找表（因为每个通道有 256 个可能值）
    # 这个表可以把任意 RGB 值映射到一个类别索引
    colormap2label = torch.zeros(256 ** 3, dtype=torch.long)
    for i, colormap in enumerate(VOC_COLORMAP):
        # 将 RGB 三元组编码为一个唯一的整数索引
        # 例如：[0, 0, 0] -> 0, [0, 0, 1] -> 1, [0, 0, 255] -> 255, [0, 1, 0] -> 256
        # 公式：(R * 256 + G) * 256 + B
        colormap2label[
            (colormap[0] * 256 + colormap[1]) * 256 + colormap[2]] = i
    return colormap2label

#@save
def voc_label_indices(colormap, colormap2label):
    """将 VOC 标签中的 RGB 值映射到它们的类别索引
    
    参数:
        colormap: 标注图像张量，形状 [C, H, W] 或 [H, W, C]，C=3
        colormap2label: voc_colormap2label() 返回的查找表
        
    返回:
        类别索引张量，形状 [H, W]，dtype=torch.long
        每个位置的值是该像素对应的类别索引 (0-20)
    """
    # 转换通道顺序：从 [C, H, W] 变为 [H, W, C]，并转为 numpy 数组
    # 转换后形状：[H, W, 3]
    colormap = colormap.permute(1, 2, 0).numpy().astype('int32')
    # 将每个像素的 RGB 值编码为一个整数
    # colormap[:, :, 0]: R 通道，形状 [H, W]
    # colormap[:, :, 1]: G 通道，形状 [H, W]
    # colormap[:, :, 2]: B 通道，形状 [H, W]
    # 计算后 idx 形状：[H, W]
    idx = ((colormap[:, :, 0] * 256 + colormap[:, :, 1]) * 256
           + colormap[:, :, 2])
    # 使用 idx 作为索引，在查找表中查找对应的类别索引
    # colormap2label[idx]: 形状 [H, W]，每个位置是该像素的类别索引 (0-20)
    return colormap2label[idx]

[**例如**]，在第一张样本图像中，飞机头部区域的类别索引为1，而背景索引为0。


In [ ]:
# 示例：查看第一张标注图像的某个局部区域的类别索引
# voc_label_indices 返回形状 [H, W] 的类别索引张量
y = voc_label_indices(train_labels[0], voc_colormap2label())
# 取出左上角 [105:115, 130:140] 区域的类别索引（10x10 的小方块）
# y[105:115, 130:140] 形状：[10, 10]，值是 0 或 1 等类别索引
# VOC_CLASSES[1] = 'aeroplane'，用于验证索引 1 确实对应飞机类别
y[105:115, 130:140], VOC_CLASSES[1]

### 预处理数据

在之前的实验，例如 :numref:`sec_alexnet`— :numref:`sec_googlenet`中，我们通过再缩放图像使其符合模型的输入形状。
然而在语义分割中，这样做需要将预测的像素类别重新映射回原始尺寸的输入图像。
这样的映射可能不够精确，尤其在不同语义的分割区域。
为了避免这个问题，我们将图像裁剪为固定尺寸，而不是再缩放。
具体来说，我们[**使用图像增广中的随机裁剪，裁剪输入图像和标签的相同区域**]。


In [ ]:
#@save
def voc_rand_crop(feature, label, height, width):
    """随机裁剪特征和标签图像
    
    语义分割中，输入图像和标注必须使用相同的裁剪参数，
    以保证裁剪后像素位置仍然一一对应
    
    参数:
        feature: 输入图像张量，形状 [C, H, W]，C=3
        label: 标注图像张量，形状 [C, H, W]，C=3
        height: 裁剪后的高度
        width: 裁剪后的宽度
        
    返回:
        feature: 裁剪后的图像，形状 [C, height, width]
        label: 裁剪后的标注，形状 [C, height, width]
    """
    # 获取随机裁剪的参数 (top, left, height, width)
    # get_params 会根据给定的 crop_size (height, width) 计算随机起始位置
    rect = torchvision.transforms.RandomCrop.get_params(
        feature, (height, width))
    # 对特征图像进行裁剪，形状从 [C, H, W] → [C, height, width]
    feature = torchvision.transforms.functional.crop(feature, *rect)
    # 对标注图像使用相同的裁剪参数，确保像素级对应关系不变
    # 形状同样从 [C, H, W] → [C, height, width]
    label = torchvision.transforms.functional.crop(label, *rect)
    return feature, label

In [ ]:
# 演示随机裁剪的效果：对同一张图像和标注进行多次随机裁剪
imgs = []
# 循环 n=5 次，每次裁剪得到 2 张图（特征图 + 标注图），共 10 张
for _ in range(n):
    # voc_rand_crop 返回 (feature_crop, label_crop)，每个形状都是 [C, 200, 300]
    imgs += voc_rand_crop(train_features[0], train_labels[0], 200, 300)

# 转换通道顺序：从 [C, H, W] → [H, W, C]，以便 matplotlib 显示
# 转换后每个图像形状：[200, 300, 3]
imgs = [img.permute(1, 2, 0) for img in imgs]
# 显示裁剪结果：2 行 n 列
# imgs[::2] 取所有偶数索引（0,2,4,6,8）→ 5 张裁剪后的原图
# imgs[1::2] 取所有奇数索引（1,3,5,7,9）→ 5 张裁剪后的标注图
d2l.show_images(imgs[::2] + imgs[1::2], 2, n);

### [**自定义语义分割数据集类**]

我们通过继承高级API提供的`Dataset`类，自定义了一个语义分割数据集类`VOCSegDataset`。
通过实现`__getitem__`函数，我们可以任意访问数据集中索引为`idx`的输入图像及其每个像素的类别索引。
由于数据集中有些图像的尺寸可能小于随机裁剪所指定的输出尺寸，这些样本可以通过自定义的`filter`函数移除掉。
此外，我们还定义了`normalize_image`函数，从而对输入图像的RGB三个通道的值分别做标准化。


In [ ]:
#@save
class VOCSegDataset(torch.utils.data.Dataset):
    """一个用于加载 VOC 数据集的自定义数据集类
    
    继承 PyTorch 的 Dataset 类，实现语义分割数据集的加载
    """

    def __init__(self, is_train, crop_size, voc_dir):
        """
        参数:
            is_train: True 为训练集，False 为验证集
            crop_size: (height, width) 裁剪后的图像尺寸
            voc_dir: VOC 数据集根目录路径
        """
        # 定义标准化变换：使用 ImageNet 的均值和标准差
        # mean=[0.485, 0.456, 0.406] 对应 RGB 三通道的均值
        # std=[0.229, 0.224, 0.225] 对应 RGB 三通道的标准差
        self.transform = torchvision.transforms.Normalize(
            mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        self.crop_size = crop_size
        # 读取所有图像和标注
        features, labels = read_voc_images(voc_dir, is_train=is_train)
        # 对特征图像进行筛选和标准化
        # 1. filter(): 移除尺寸小于裁剪尺寸的图像
        # 2. normalize_image(): 将像素值从 [0, 255] 归一化到 [0, 1]，再标准化
        # self.features: 筛选并标准化后的图像列表，每个形状 [3, H, W]
        self.features = [self.normalize_image(feature)
                         for feature in self.filter(features)]
        # 对标注进行筛选（移除尺寸过小的），但不做标准化
        # self.labels: 筛选后的标注列表，每个形状 [3, H, W]
        self.labels = self.filter(labels)
        # 构建 RGB 到类别索引的查找表
        self.colormap2label = voc_colormap2label()
        print('read ' + str(len(self.features)) + ' examples')

    def normalize_image(self, img):
        """将图像像素值归一化并标准化
        参数:
            img: 输入图像张量，形状 [C, H, W]，dtype=uint8, 范围 [0, 255]
        返回:
            标准化后的图像，形状 [C, H, W]，dtype=float32
        """
        # img.float() / 255: 将 uint8 [0, 255] → float32 [0, 1]
        # self.transform(): 使用 ImageNet 均值和标准差标准化
        return self.transform(img.float() / 255)

    def filter(self, imgs):
        """过滤掉尺寸小于裁剪尺寸的图像
        参数:
            imgs: 图像列表
        返回:
            过滤后的图像列表
        """
        # img.shape[1]: 高度 H, img.shape[2]: 宽度 W
        # 只保留 H >= crop_size[0] 且 W >= crop_size[1] 的图像
        return [img for img in imgs if (
            img.shape[1] >= self.crop_size[0] and
            img.shape[2] >= self.crop_size[1])]

    def __getitem__(self, idx):
        """获取索引为 idx 的样本
        
        参数:
            idx: 样本索引
            
        返回:
            feature: 裁剪并标准化后的图像，形状 [3, height, width]
            label: 裁剪后的类别索引，形状 [height, width]，dtype=torch.long
        """
        # 随机裁剪特征和标注，保持像素对应关系
        # feature 形状：[3, crop_height, crop_width]
        # label 形状：[3, crop_height, crop_width]，RGB 编码
        feature, label = voc_rand_crop(self.features[idx], self.labels[idx],
                                       *self.crop_size)
        # 将 label 从 RGB 编码转换为类别索引
        # voc_label_indices 返回形状 [height, width]，每个值是 0-20 的类别索引
        return (feature, voc_label_indices(label, self.colormap2label))

    def __len__(self):
        """返回数据集大小"""
        return len(self.features)

### [**读取数据集**]

我们通过自定义的`VOCSegDataset`类来分别创建训练集和测试集的实例。
假设我们指定随机裁剪的输出图像的形状为$320\times 480$，
下面我们可以查看训练集和测试集所保留的样本个数。


In [ ]:
# 指定随机裁剪后的输出图像尺寸：高 320 像素，宽 480 像素
crop_size = (320, 480)
# 创建训练集实例：会读取图像、筛选、标准化，并打印保留的样本数
voc_train = VOCSegDataset(True, crop_size, voc_dir)
# 创建验证集实例
voc_test = VOCSegDataset(False, crop_size, voc_dir)

设批量大小为64，我们定义训练集的迭代器。
打印第一个小批量的形状会发现：与图像分类或目标检测不同，这里的标签是一个三维数组。


In [ ]:
# 设置批量大小为 64，即每次迭代返回 64 个样本
batch_size = 64
# 创建训练集的 DataLoader
# shuffle=True: 每个 epoch 打乱数据顺序
# drop_last=True: 丢弃最后一个不完整的 batch
# num_workers: 使用多进程加载数据，加速数据读取
train_iter = torch.utils.data.DataLoader(voc_train, batch_size, shuffle=True,
                                    drop_last=True,
                                    num_workers=d2l.get_dataloader_workers())
# 遍历训练数据迭代器，取出第一个批量进行形状检查
for X, Y in train_iter:
    # X: 图像批量，形状 [batch_size, C, H, W] = [64, 3, 320, 480]
    #   64 个样本，每样本 3 通道 (RGB)，高 320 像素，宽 480 像素
    print(X.shape)
    # Y: 标注批量，形状 [batch_size, H, W] = [64, 320, 480]
    #   64 个样本，每个样本是一个 320x480 的类别索引矩阵
    #   注意：与图像分类不同，语义分割的标签是二维的（每个像素一个类别）
    print(Y.shape)
    break  # 只检查第一个批量

### [**整合所有组件**]

最后，我们定义以下`load_data_voc`函数来下载并读取Pascal VOC2012语义分割数据集。
它返回训练集和测试集的数据迭代器。


In [ ]:
#@save
def load_data_voc(batch_size, crop_size):
    """加载 VOC 语义分割数据集
    
    参数:
        batch_size: 批量大小
        crop_size: (height, width) 裁剪尺寸
        
    返回:
        train_iter: 训练集 DataLoader
        test_iter: 测试集 DataLoader
    """
    # 下载并提取 VOC2012 数据集，返回数据集根目录
    voc_dir = d2l.download_extract('voc2012', os.path.join(
        'VOCdevkit', 'VOC2012'))
    # 获取数据加载的工作进程数
    num_workers = d2l.get_dataloader_workers()
    # 创建训练集 DataLoader
    # VOCSegDataset(True, ...): 训练集，会进行数据增强（随机裁剪）
    train_iter = torch.utils.data.DataLoader(
        VOCSegDataset(True, crop_size, voc_dir), batch_size,
        shuffle=True, drop_last=True, num_workers=num_workers)
    # 创建测试集 DataLoader
    # VOCSegDataset(False, ...): 验证集，不做数据增强
    test_iter = torch.utils.data.DataLoader(
        VOCSegDataset(False, crop_size, voc_dir), batch_size,
        drop_last=True, num_workers=num_workers)
    return train_iter, test_iter

## 小结

* 语义分割通过将图像划分为属于不同语义类别的区域，来识别并理解图像中像素级别的内容。
* 语义分割的一个重要的数据集叫做Pascal VOC2012。
* 由于语义分割的输入图像和标签在像素上一一对应，输入图像会被随机裁剪为固定尺寸而不是缩放。

## 练习

1. 如何在自动驾驶和医疗图像诊断中应用语义分割？还能想到其他领域的应用吗？
1. 回想一下 :numref:`sec_image_augmentation`中对数据增强的描述。图像分类中使用的哪种图像增强方法是难以用于语义分割的？


[Discussions](https://discuss.d2l.ai/t/3295)
